# TractorMixGenome post-workflow QC / PheWAS views

Consumes **`TractorMixGenome`** summary outputs (not chrom shards):

| Workflow output | Use |
|-----------------|-----|
| `results_manifest` | Phenotype → named TSV mapping |
| `results_tsvs_named` / `results_by_phenotype/*.tsv` | Association tables |
| `calibration_summary` | λGC, N cases/controls, hit counts |
| `phewas_genomewide_hits` | Stacked genome-wide hits |
| `qq_plots` / `manhattan_plots` | Per-trait figures from the WDL |

This notebook adds **cross-phenotype** views (λ barplot, hit counts, PheWAS heatmap)
and optional **limited vs full** λ comparison. Leave LDSC / VEP / portal export for
later methods — they need external references.

Set `GENOME_SUMMARY_GCS` to a submission prefix that contains the Summarize task
outputs, or point `LOCAL_SUMMARY_DIR` at a localized copy.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

for _d in (Path.cwd() / "scripts", Path.cwd().parent / "scripts"):
    if (_d / "terra_notebook.py").is_file():
        sys.path.insert(0, str(_d.resolve()))
        break
else:
    _bucket = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
    if not _bucket:
        raise FileNotFoundError("scripts/ missing and WORKSPACE_BUCKET unset")
    _dest = (Path.cwd() / "scripts").resolve()
    _dest.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(
        ["gsutil", "-m", "rsync", "-r", f"{_bucket}/scripts/", str(_dest) + "/"]
    )
    sys.path.insert(0, str(_dest))

from terra_notebook import init_notebook

SCRIPTS = init_notebook(
    "workspace_paths.py",
    "genome_post_workflow.py",
    "summarize_tractor_genome_results.py",
)
from genome_post_workflow import (
    compare_lambda_two_models,
    load_calibration,
    load_manifest,
    phewas_hit_counts,
    phewas_lambda_bar,
    phewas_locus_heatmap,
)

OUT = Path(os.environ.get("GENOME_POST_QC_DIR", "genome_post_qc"))
OUT.mkdir(parents=True, exist_ok=True)
LOCAL = Path(os.environ.get("LOCAL_SUMMARY_DIR", "genome_summary_inputs"))
LOCAL.mkdir(parents=True, exist_ok=True)

# Example:
#   export GENOME_SUMMARY_GCS='gs://BUCKET/submissions/.../call-Summarize'
GENOME_SUMMARY_GCS = os.environ.get("GENOME_SUMMARY_GCS", "").rstrip("/")
# Optional second model for limited-vs-full λ comparison
GENOME_SUMMARY_GCS_FULL = os.environ.get("GENOME_SUMMARY_GCS_FULL", "").rstrip("/")

ws = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")


def sh(cmd: str) -> None:
    print(cmd)
    rc = get_ipython().system(cmd)
    if rc:
        raise RuntimeError(f"command failed with exit code {rc}: {cmd}")


print("OUT:", OUT.resolve())
print("LOCAL:", LOCAL.resolve())
print("GENOME_SUMMARY_GCS:", GENOME_SUMMARY_GCS or "(unset — use LOCAL files)")

In [ ]:
def pull_summary(gcs_prefix: str, dest: Path) -> Path:
    """Localize SummarizeGenomeResults outputs (flat or nested under summary/)."""
    dest.mkdir(parents=True, exist_ok=True)
    if not gcs_prefix:
        return dest
    # Prefer recursive pull of the Summarize execution directory
    sh(f"gsutil -m rsync -r {gcs_prefix}/ {dest}/")
    return dest


def find_summary_root(base: Path) -> Path:
    """Return directory containing results_manifest.tsv."""
    for d in (base, base / "summary"):
        if (d / "results_manifest.tsv").is_file():
            return d
    matches = sorted(base.glob("**/results_manifest.tsv"))
    if matches:
        return matches[0].parent
    raise FileNotFoundError(
        f"results_manifest.tsv not under {base}. Set GENOME_SUMMARY_GCS or LOCAL_SUMMARY_DIR."
    )


pull_summary(GENOME_SUMMARY_GCS, LOCAL)
SUMMARY = find_summary_root(LOCAL)
print("Using summary root:", SUMMARY)

manifest = load_manifest(SUMMARY / "results_manifest.tsv")
calib = load_calibration(SUMMARY / "calibration_summary.tsv")
display(manifest)
display(calib)

In [ ]:
from IPython.display import Image, Markdown, display
import pandas as pd

md = SUMMARY / "calibration_summary.md"
if md.exists():
    display(Markdown(md.read_text()))

phewas_lambda_bar(calib, OUT / "lambda_by_phenotype.png")
phewas_hit_counts(calib, OUT / "hits_by_phenotype.png")
display(Image(filename=str(OUT / "lambda_by_phenotype.png")))
display(Image(filename=str(OUT / "hits_by_phenotype.png")))

hits_path = SUMMARY / "phewas_genomewide_hits.tsv"
hits = pd.read_csv(hits_path, sep="\t") if hits_path.exists() else pd.DataFrame()
print(f"genome-wide hit rows: {len(hits):,}")
if not hits.empty:
    display(hits.head(20))
    phewas_locus_heatmap(hits, OUT / "phewas_locus_heatmap.png")
    display(Image(filename=str(OUT / "phewas_locus_heatmap.png")))

In [ ]:
# Browse WDL-generated per-trait QQ / Manhattan (if localized)
qc_root = SUMMARY / "qc"
if not qc_root.exists():
    # Flat glob layout from Terra Array[File] downloads
    print("No qc/ tree; showing any downloaded png next to summary root")
    for png in sorted(SUMMARY.glob("**/*manhattan*.png"))[:5]:
        print(png)
        display(Image(filename=str(png)))
else:
    for pheno_dir in sorted(p for p in qc_root.iterdir() if p.is_dir())[:5]:
        print("===", pheno_dir.name, "===")
        for name in ("qq_joint_acpass.png", "manhattan_joint.png"):
            png = pheno_dir / name
            if png.exists():
                display(Image(filename=str(png)))
        top = pheno_dir / "top_hits.tsv"
        if top.exists():
            display(pd.read_csv(top, sep="\t").head(10))

In [ ]:
# Optional: limited vs full covariate λGC comparison
if GENOME_SUMMARY_GCS_FULL:
    LOCAL_FULL = Path("genome_summary_inputs_full")
    pull_summary(GENOME_SUMMARY_GCS_FULL, LOCAL_FULL)
    SUMMARY_FULL = find_summary_root(LOCAL_FULL)
    calib_full = load_calibration(SUMMARY_FULL / "calibration_summary.tsv")
    cmp = compare_lambda_two_models(
        calib, calib_full, OUT / "lambda_limited_vs_full.png"
    )
    cmp.to_csv(OUT / "lambda_limited_vs_full.tsv", sep="\t", index=False)
    display(cmp)
    display(Image(filename=str(OUT / "lambda_limited_vs_full.png")))
else:
    print("Set GENOME_SUMMARY_GCS_FULL to compare limited vs full λGC.")

## Next analyses (not in this notebook)

- **LDSC / heritability / genetic correlation** — needs LD reference + munged sumstats
- **Variant annotation (VEP / nearest gene)** — separate annotation WDL on lead SNPs
- **SAIGE matched calibration** — reuse `tractor_02_qc_results.ipynb` / `compare_calibration.py`
- **Fine-mapping / conditional** — per-locus follow-up after hit calling

Upload notebook figures back to the workspace if desired:

```bash
gsutil -m cp -r genome_post_qc "$WORKSPACE_BUCKET/tractor_mix_genome_qc/"
```